In [ ]:
import os

REPO = '/content/docstring-generator-cli'
CLONE_URL = 'https://github.com/brownricee/docstring-generator-cli.git'

if os.path.exists(REPO):
    # A clone from an earlier session is already here. `git clone` would fail
    # and leave the OLD code in place, so update it instead.
    !cd {REPO} && git fetch origin && git pull --ff-only origin main
else:
    !git clone {CLONE_URL} {REPO}

# Print what we are actually about to run. If this is not the commit you
# expect, nothing below is worth running.
!cd {REPO} && git log --oneline -1


In [ ]:
%cd /content/docstring-generator-cli


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
assert os.path.ismount('/content/drive'), 'Drive did not mount -- do not continue'

DATA_DIR = '/content/drive/MyDrive/docstring_generator_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('checkpoints ->', DATA_DIR)


In [ ]:
import os

os.environ['CHECKPOINT_DIR'] = '/content/drive/MyDrive/docstring_generator_data'

# Must be set before the training process starts torch. `!` commands inherit
# this kernel's environment, so setting it here reaches the subprocess below.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [ ]:
!python -m training.train --sanity


In [ ]:
# -u is load-bearing. Piping into `tee` makes stdout a pipe rather than a tty, so
# CPython block-buffers it at 8KB and the cell shows nothing for ~20 minutes while
# the training prints pile up in the buffer. -u disables that buffering.
# tee writes to local disk, not the Drive FUSE mount -- appending a line every few
# seconds over FUSE is a lot of round-trips for no benefit. The next cell copies it.
!python -u -m training.train 2>&1 | tee /content/train_log.txt


In [ ]:
# Checkpoints already go straight to Drive; the log only gets there if the cell
# above finishes. A killed session loses the log but keeps the weights.
!cp /content/train_log.txt "$CHECKPOINT_DIR/"
!ls -la "$CHECKPOINT_DIR"


In [ ]:
# Get the adapter from the published release instead of Drive -- no mount
# needed just to run eval.
!mkdir -p checkpoints
!curl -L -o checkpoints/lora_weights.pt https://github.com/brownricee/docstring-generator-cli/releases/download/adapter-v1/lora_weights.pt


In [ ]:
# -u avoids the same buffering issue noted above -- piped stdout is
# block-buffered without it.
!python -u -m training.evaluate --n 100 2>&1 | tee /content/eval_log.txt


In [ ]:
# The GGUF passes run on CPU, so a CPU wheel is fine even on a GPU runtime.
# PyPI has only an sdist for llama-cpp-python (would compile for many minutes);
# the extra index serves a prebuilt wheel.
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu


In [ ]:
# Fetch the Q8_0 GGUF. Needs the gguf-v1 release asset to exist; if you have not
# uploaded it yet, copy it from Drive instead (see the commented line).
!mkdir -p checkpoints
!curl -fL -o checkpoints/docgen-q8_0.gguf https://github.com/brownricee/docstring-generator-cli/releases/download/gguf-v1/docgen-q8_0.gguf
# !cp "$CHECKPOINT_DIR/docgen-q8_0.gguf" checkpoints/
!ls -la checkpoints


In [ ]:
# Torch fine-tuned (reference) vs Q8_0 GGUF on the same 100 fixed-seed samples.
# Prints a table: pass-rate for each model, and how many Q8_0 outputs are
# identical to the torch model's after cleanup. These are the README numbers.
# Needs checkpoints/lora_weights.pt from the adapter cell above.
!python -u -m training.evaluate --n 100 --gguf checkpoints/docgen-q8_0.gguf 2>&1 | tee /content/gguf_eval_log.txt
